<a href="https://colab.research.google.com/github/Yike-Ding/structural_bioinformatics/blob/main/02_binding_site_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**02 Binding Site Identification**

The histone family is a key component of nucleosome. It plays a central role in transcription regulation, DNA replication and chromosomal stability.

With the isolated histone 3.1 structure obtained, we examine whether it is a suitable target by evaluating potential binding sites.

In [1]:
import numpy
import matplotlib.pyplot
import os
import requests


In [ ]:
import os
import urllib.request

# 替换为你自己的 GitHub 用户名、仓库名、分支名及文件路径
GITHUB_RAW_PDB_URL = "https://raw.githubusercontent.com/你的用户名/你的仓库名/main/data/my_processed_protein.pdb"
pdb_filename = "my_processed_protein.pdb"

if not os.path.exists(pdb_filename):
  print(f"Downloading {pdb_filename} from GitHub...")
  urllib.request.urlretrieve(GITHUB_RAW_PDB_URL, pdb_filename)
  print("Download completed!")
else:
  print(f"{pdb_filename} already exists.")

Assign a pdb code from our customized pdb file at the protein plus site.

In [ ]:
def pdb_file_to_pdb_code(pdb_path: str) -> str:
    """Upload a local PDB file to ProteinsPlus and retrieve a temporary PDB code

    (ID) that can be passed to subsequent tools (like DoGSite3).

    Parameters
    ----------
    pdb_path : str
        Path to the local processed PDB file (e.g., 'protein_processed.pdb').

    Returns
    -------
    str
        Temporary PDB identifier assigned by ProteinsPlus.
    """
    if not os.path.exists(pdb_path):
        raise FileNotFoundError(f"PDB file not found: {pdb_path}")

    url = "https://proteins.plus/api/pdb_files_rest"
    headers = {"Accept": "application/json"}

    # Use multipart/form-data via the 'files' parameter
    # Do not set 'Content-Type' manually; requests handles boundary generation
    with open(pdb_path, "rb") as f:
        files = {"pdb_file[pathvar]": (os.path.basename(pdb_path), f)}
        response = requests.post(url, files=files, headers=headers)

    response.raise_for_status()
    res_data = response.json()

    # Extract the generated PDB ID
    if "id" in res_data:
        return res_data["id"]
    else:
        raise KeyError(f"Missing 'id' field in response: {res_data}")

In [ ]:
def submit_dogsite3_job_with_pdbid(pdb_code: str, chain_id: str, ligand: str = ""):
  """
  Submit pdb code to DoGSiteScorer API and get back url.

  Parameters
  ---------
  pdb file

  Returns
  ---------
  str
        The job location URL (relative ID) to poll for results.
  """
  url = "https://proteins.plus/api/dogsite3_rest"
  payload = {
      dogsite3 :{
        "pdbCode":pdbCode,
        "analysisDetail" : "0",
        "bindingSitePredictionGranularity": "1",
        "ligand": ligand,
        "chain":chain_id,
        "ligandBias": "0"
    }
  }
  Header = {"Accept": application/json, "Content-Type": application/json}

  r = request.post()
  r.raise_for_status()
  resp = r.json()

  # Should return status_code + location (job ID)
  if "location" in resp:
      return resp["location"]
  else:
      raise RuntimeError("Unexpected response from DoGSite3 submission: %r" % resp)


_IncompleteInputError: incomplete input (3218091819.py, line 2)